In [ ]:
# Import necessary ROS2 libraries
import rclpy
from rclpy.node import Node
import time # For potential delays or checks
import sys
import os
import cv2
import json
from ament_index_python.packages import get_package_share_directory

from utils.vision_functions import VisionHandler

# from pm_vision_manager.va_py_modules.vision_pipline_processing import process_image, set_camera_parameters

# try:
#     from pm_vision_manager.va_py_modules.vision_assistant_class import VisionProcessClass
# except ImportError:
#     # If the direct import fails, try to add the package path
#     # This is a common workaround for development in Jupyter
#     ros2_ws_src_path = os.path.expanduser('~/ros2_ws/src') # Adjust to your workspace src directory
#     my_vision_package_path = os.path.join(ros2_ws_src_path, 'my_vision_package', 'my_vision_package')
#     if os.path.exists(my_vision_package_path):
#         sys.path.insert(0, os.path.dirname(my_vision_package_path))
#         from pm_vision_manager.va_py_modules.vision_assistant_class import VisionProcessClass
#     else:
#         print(f"Could not find my_vision_package at {my_vision_package_path}. Please check your path or install the package.")
#         raise

# --- ROS2 Node Initialization and Usage ---

# Initialize ROS2 (only call once per process)
if not rclpy.ok():
    rclpy.init()
    print("ROS2 initialized.")
else:
    print("ROS2 already initialized.")

# Create a simple ROS2 node
# This node will be passed to your VisionProcessClass
node = None
try:
    node = Node('jupyter_vision_test_node')
    node.get_logger().info('Jupyter ROS2 node created.')

    vision_handler = VisionHandler(node) 

    vision_functions = vision_handler.return_available_function_names()
    # node.get_logger().info(f"Available vision functions: {vision_functions}")

    process_file_name = "grayscale_threshold.json"

    try:
        result_dict, message = vision_handler.process_image(
            image_name="cube.png",  # Replace with actual image name
            process_filename=process_file_name
        )
    except Exception as e:
        print(f"Error processing image: {e}")

except Exception as e:
    if node:
        node.get_logger().error(f'An error occurred: {e}')
    else:
        print(f'An error occurred before node creation: {e}')
finally:
    # Always destroy the node and shutdown ROS2 when you are done.
    # This is crucial for proper resource management and to avoid errors on re-execution.
    if node:
        node.destroy_node()
        print("ROS2 node destroyed.")
    if rclpy.ok(): # Check if ROS2 is still active before shutting down
        rclpy.shutdown()
        print("ROS2 shutdown.")

print("Jupyter Notebook ROS2 test completed.")



ROS2 initialized.
ROS2 node destroyed.
ROS2 shutdown.
Jupyter Notebook ROS2 test completed.


[INFO] [1753783266.198782693] [jupyter_vision_test_node]: Jupyter ROS2 node created.
[INFO] [1753783266.232030525] [jupyter_vision_test_node]: Parameters for CircleDetection: CircleDetection


In [ ]:
# Import necessary ROS2 libraries
import rclpy
from rclpy.node import Node
import time # For potential delays or checks
import sys
import os
import cv2

from pm_co_pilot_vision.utils.vision_functions import VisionFunctions

from pm_vision_manager.va_py_modules.vision_pipline_processing import process_image, set_camera_parameters

try:
    from pm_vision_manager.va_py_modules.vision_assistant_class import VisionProcessClass
except ImportError:
    # If the direct import fails, try to add the package path
    # This is a common workaround for development in Jupyter
    ros2_ws_src_path = os.path.expanduser('~/ros2_ws/src') # Adjust to your workspace src directory
    my_vision_package_path = os.path.join(ros2_ws_src_path, 'my_vision_package', 'my_vision_package')
    if os.path.exists(my_vision_package_path):
        sys.path.insert(0, os.path.dirname(my_vision_package_path))
        from pm_vision_manager.va_py_modules.vision_assistant_class import VisionProcessClass
    else:
        print(f"Could not find my_vision_package at {my_vision_package_path}. Please check your path or install the package.")
        raise

# --- ROS2 Node Initialization and Usage ---

# Initialize ROS2 (only call once per process)
if not rclpy.ok():
    rclpy.init()
    print("ROS2 initialized.")
else:
    print("ROS2 already initialized.")

# Create a simple ROS2 node
# This node will be passed to your VisionProcessClass
node = None
try:
    node = Node('jupyter_vision_test_node')
    node.get_logger().info('Jupyter ROS2 node created.')

    # Define dummy filenames for testing
    process_filename = "test_pipeline.json" # Replace with actual path
    camera_config_filename = "pm_robot_top_cam_1_Unity.yaml" # Replace with actual path
    image_path = "/home/match-mover/pm_Server/01_PM_Zelle/03_PM_DataBase/pm_assembly_database/Vision/vision_image_db/test_pipeline/"
    image_name = "target.png"

    # load image as image:np.ndarray
    image = cv2.imread(f"{image_path}/{image_name}")

    # Instantiate your VisionProcessClass
    vision_processor = VisionProcessClass(
        vision_node=node,
        process_filename=process_filename,
        camera_config_filename=camera_config_filename,
        launch_as_assistant=False, # Set as per your testing needs
        process_UID='JupyterTestVision',
        run_cross_validation=False # Set as per your testing needs
    )

    node.get_logger().info('VisionProcessClass instantiated.')

    vision_processor._load_process_file()

    set_parameter_success = set_camera_parameters(vision_processor.vision_node, 
                                                    vision_processor.image_processing_handler,
                                                    vision_processor.process_pipeline_list)

    if not set_parameter_success:
        vision_processor.image_processing_handler.vision_routine_done = True
        vision_processor.image_processing_handler.visionOK = False
        # error
        node.get_logger().error('Failed to set camera parameters. Exiting...')
        rclpy.shutdown()
        sys.exit(1)
    
    node.get_logger().info('Camera parameters set successfully.')
        
    vision_processor.image_processing_handler.set_image_metatdata(vision_processor.process_db_path,image_name)
    vision_processor.image_processing_handler.set_initial_image(image)

    #display_image = process_image(self, image, self.process_pipeline_list)
    display_image, vision_results = process_image(vision_processor.vision_node, vision_processor.image_processing_handler, vision_processor.process_pipeline_list)
    final_image = vision_processor.image_processing_handler.get_final_image()

    node.get_logger().info(f'vision results: {vision_results}')

    result_dict = vision_processor.save_vision_results()

    node.get_logger().info(f'vision result: {result_dict}')

    # save final image
    image_name_processed = image_name.replace('.png', '_processed.png')
    cv2.imwrite(f"{image_path}/final_{image_name_processed}", final_image)


except Exception as e:
    if node:
        node.get_logger().error(f'An error occurred: {e}')
    else:
        print(f'An error occurred before node creation: {e}')
finally:
    # Always destroy the node and shutdown ROS2 when you are done.
    # This is crucial for proper resource management and to avoid errors on re-execution.
    if node:
        node.destroy_node()
        print("ROS2 node destroyed.")
    if rclpy.ok(): # Check if ROS2 is still active before shutting down
        rclpy.shutdown()
        print("ROS2 shutdown.")

print("Jupyter Notebook ROS2 test completed.")

ROS2 initialized.


[INFO] [1753712081.674133716] [jupyter_vision_test_node]: Jupyter ROS2 node created.
[INFO] [1753712081.733665100] [jupyter_vision_test_node]: Execute Vision!
[INFO] [1753712081.734142761] [jupyter_vision_test_node]: Current vision process: test_pipeline.json
[INFO] [1753712081.735660802] [jupyter_vision_test_node]: Vision assistant path config loaded!
[INFO] [1753712081.736436293] [jupyter_vision_test_node]: Vision assistant config loaded!
[INFO] [1753712081.739583717] [jupyter_vision_test_node]: Process meta data loaded!
[WARN] [1753712081.745036977] [jupyter_vision_test_node]: Camera exposure_time not defined!
[WARN] [1753712082.763140060] [jupyter_vision_test_node]: Set coax light bool client not online!
[ERROR] [1753712082.766385441] [jupyter_vision_test_node]: Exception occured
[ERROR] [1753712082.768132723] [jupyter_vision_test_node]: CameraExposureTimeError
[ERROR] [1753712082.769800236] [jupyter_vision_test_node]: Camera exposure time service is not available!
[WARN] [17537120

Unique values: [  0 255]
Unique values: [  0 255]
ROS2 node destroyed.
ROS2 shutdown.
Jupyter Notebook ROS2 test completed.
